# Exploratory Data Analysis - Online Retail\n
\n
Tujuan notebook ini:\n
- Audit kualitas data dan anomali (mis. quantity negatif).\n
- EDA menyeluruh untuk memahami pola demand.\n
- Menyusun pipeline cleaning untuk kebutuhan demand forecasting (per hari per produk).\n
\n
Catatan: notebook hanya menampilkan hasil data bersih (preview/statistik), tidak menyimpan ke disk.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 120)
sns.set_theme(style="whitegrid")

In [ ]:
data_path = "../data/raw/online_retail.csv"
df_raw = pd.read_csv(data_path)
df_raw.head()

In [ ]:
df_raw.shape

## Audit Kualitas Data

In [ ]:
df_raw.info()

In [ ]:
missing_summary = df_raw.isna().sum().sort_values(ascending=False)
missing_summary

In [ ]:
duplicate_rows = df_raw.duplicated().sum()
duplicate_rows

### Deteksi Anomali dan Nilai Tidak Masuk Akal\n
Contoh anomali: quantity negatif, price nol/negatif, InvoiceDate invalid, atau StockCode kosong.

In [ ]:
anomaly_summary = pd.DataFrame([{
    "quantity_le_0": (df_raw["Quantity"] <= 0).sum(),
    "price_le_0": (df_raw["Price"] <= 0).sum(),
    "missing_invoice_date": df_raw["InvoiceDate"].isna().sum(),
    "missing_stock_code": df_raw["StockCode"].isna().sum(),
    "missing_invoice": df_raw["Invoice"].isna().sum(),
}])
anomaly_summary

## Pipeline Cleaning (Target: Demand per Hari per Produk)

In [ ]:
def clean_online_retail(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()

    df = df.rename(columns={
        "Invoice": "invoice",
        "StockCode": "stock_code",
        "Description": "description",
        "Quantity": "quantity",
        "InvoiceDate": "invoice_date",
        "Price": "price",
        "Customer ID": "customer_id",
        "Country": "country",
    })

    df["invoice_date"] = pd.to_datetime(df["invoice_date"], errors="coerce")
    df["quantity"] = pd.to_numeric(df["quantity"], errors="coerce")
    df["price"] = pd.to_numeric(df["price"], errors="coerce")

    df["stock_code"] = df["stock_code"].astype(str).str.strip().str.upper()
    df["invoice"] = df["invoice"].astype(str).str.strip()

    df = df.drop_duplicates()

    df = df[df["invoice_date"].notna()]
    df = df[df["stock_code"].notna() & (df["stock_code"] != "")]
    df = df[df["invoice"].notna() & (df["invoice"] != "")]

    df = df[~df["invoice"].str.startswith("C", na=False)]

    non_product_codes = {
        "POST",
        "DOT",
        "C2",
        "M",
        "D",
        "ADJUST",
        "ADJUST2",
        "BANK CHARGES",
        "AMAZONFEE",
        "B",
        "S",
        "PADS",
        "TEST001",
        "TEST002",
        "GIFT_0001_10",
        "GIFT_0001_20",
        "GIFT_0001_30",
        "GIFT_0001_40",
        "GIFT_0001_50",
        "GIFT_0001_70",
        "GIFT_0001_80",
    }
    df = df[~df["stock_code"].isin(non_product_codes)]

    df = df[(df["quantity"] > 0) & (df["price"] > 0)]

    df["revenue"] = df["quantity"] * df["price"]

    return df

df_clean = clean_online_retail(df_raw)
df_clean.shape

In [ ]:
df_clean.head()

In [ ]:
# tampilkan baris dengan stock_code non-angka dari df_clean
mask_non_numeric = ~df_clean["stock_code"].astype(str).str.match(r"^\d+$")
non_numeric_stockcodes = df_clean[mask_non_numeric]
non_numeric_stockcodes

In [ ]:
mask_invoice_c = df_clean["invoice"].astype(str).str.startswith("C", na=False)
invoice_c_rows = df_clean.loc[mask_invoice_c].copy()

invoice_c_rows.head(50)

In [ ]:
pattern = r'^\d+[A-Za-z]?$'
mask_invalid = ~df_clean['stock_code'].astype(str).str.match(pattern)

invalid = df_clean.loc[mask_invalid, ['stock_code', 'description']].copy()
invalid_summary = (
    invalid.groupby('stock_code', as_index=False)
    .agg(rows=('stock_code', 'size'), sample_description=('description', 'first'))
    .sort_values('rows', ascending=False)
)

print(f"Total baris tidak cocok: {len(invalid)}")
print(f"Unique stock_code tidak cocok: {invalid_summary.shape[0]}")
invalid_summary.head(200)

In [ ]:
# Filter df_clean untuk stock_code yang tidak cocok dengan pola ^\d+[A-Za-z]?$
pattern = r'^\d+[A-Za-z]?$'
mask_invalid_pattern = ~df_clean['stock_code'].astype(str).str.match(pattern)

invalid_stockcodes = df_clean[mask_invalid_pattern][['stock_code', 'description', 'quantity', 'revenue']].copy()
invalid_summary = (
    invalid_stockcodes.groupby('stock_code', as_index=False)
    .agg(
        rows=('stock_code', 'size'),
        sample_description=('description', 'first'),
        total_qty=('quantity', 'sum'),
        total_revenue=('revenue', 'sum')
    )
    .sort_values('rows', ascending=False)
)

print(f"Total baris dengan stock_code tidak cocok: {len(invalid_stockcodes)}")
print(f"Unique stock_code tidak cocok: {invalid_summary.shape[0]}\n")
invalid_summary

In [ ]:
df_clean.loc[df_clean["stock_code"] == "PADS"]

## EDA - Distribusi dan Ringkasan

In [ ]:
df_clean[["quantity", "price", "revenue"]].describe(percentiles=[0.5, 0.9, 0.95, 0.99])

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 4))
sns.boxplot(x=df_clean["quantity"], ax=axes[0])
axes[0].set_title("Quantity Box Plot")

sns.boxplot(x=df_clean["price"], ax=axes[1])
axes[1].set_title("Price Box Plot")

sns.boxplot(x=df_clean["revenue"], ax=axes[2])
axes[2].set_title("Revenue Box Plot")

plt.tight_layout()

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 4))
sns.boxplot(x=df_clean["quantity"], ax=axes[0])
axes[0].set_xscale("log")
axes[0].set_title("Quantity Box Plot (Log Scale)")

sns.boxplot(x=df_clean["price"], ax=axes[1])
axes[1].set_xscale("log")
axes[1].set_title("Price Box Plot (Log Scale)")

sns.boxplot(x=df_clean["revenue"], ax=axes[2])
axes[2].set_xscale("log")
axes[2].set_title("Revenue Box Plot (Log Scale)")

plt.tight_layout()

## Top Produk dan Negara

In [ ]:
top_products = (
    df_clean.groupby("stock_code", as_index=False)
    .agg(demand_qty=("quantity", "sum"), revenue=("revenue", "sum"))
    .sort_values("demand_qty", ascending=False)
    .head(10)
)
top_products

In [ ]:
top_countries = (
    df_clean.groupby("country", as_index=False)
    .agg(demand_qty=("quantity", "sum"), revenue=("revenue", "sum"))
    .sort_values("demand_qty", ascending=False)
    .head(10)
)
top_countries

## Tren Waktu

In [ ]:
df_clean["date"] = df_clean["invoice_date"].dt.date
daily_trend = (
    df_clean.groupby("date", as_index=False)
    .agg(demand_qty=("quantity", "sum"), revenue=("revenue", "sum"))
)

plt.figure(figsize=(12, 4))
sns.lineplot(data=daily_trend, x="date", y="demand_qty")
plt.title("Daily Demand Trend")
plt.tight_layout()

In [ ]:
df_clean["day_of_week"] = df_clean["invoice_date"].dt.day_name()
dow = (
    df_clean.groupby("day_of_week", as_index=False)
    .agg(demand_qty=("quantity", "sum"))
)
dow["day_of_week"] = pd.Categorical(dow["day_of_week"], categories=[
    "Monday", "Tuesday", "Wednesday", "Thursday", "Friday", "Saturday", "Sunday"
], ordered=True)
dow = dow.sort_values("day_of_week")

plt.figure(figsize=(8, 4))
sns.barplot(data=dow, x="day_of_week", y="demand_qty")
plt.title("Demand by Day of Week")
plt.xticks(rotation=30)
plt.tight_layout()

## Dataset Harian per Produk (Siap untuk Forecasting)

In [ ]:
def build_daily_product_dataset(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    df["date"] = df["invoice_date"].dt.normalize()

    daily = (
        df.groupby(["stock_code", "date"], as_index=False)
        .agg(
            demand_qty=("quantity", "sum"),
            revenue=("revenue", "sum"),
            num_invoices=("invoice", "nunique"),
        )
    )

    stock_codes = daily["stock_code"].unique()
    full_dates = pd.date_range(daily["date"].min(), daily["date"].max(), freq="D")
    full_index = pd.MultiIndex.from_product([stock_codes, full_dates], names=["stock_code", "date"])

    daily = (
        daily.set_index(["stock_code", "date"])
        .reindex(full_index, fill_value=0)
        .reset_index()
    )

    return daily

daily_product = build_daily_product_dataset(df_clean)
daily_product.head()

## Dataset untuk Model Time Series dan Tabular

In [ ]:
def add_time_features(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    df["date"] = pd.to_datetime(df["date"])
    df["day_of_week"] = df["date"].dt.dayofweek
    df["week_of_year"] = df["date"].dt.isocalendar().week.astype(int)
    df["month"] = df["date"].dt.month
    df["quarter"] = df["date"].dt.quarter
    df["day_of_month"] = df["date"].dt.day
    df["is_weekend"] = (df["day_of_week"] >= 5).astype(int)
    df["is_month_start"] = df["date"].dt.is_month_start.astype(int)
    df["is_month_end"] = df["date"].dt.is_month_end.astype(int)
    return df

def add_lag_rolling_features(df: pd.DataFrame, group_col: str) -> pd.DataFrame:
    df = df.copy()
    df = df.sort_values([group_col, "date"])

    for lag in [1, 7, 14, 28]:
        df[f"lag_{lag}"] = df.groupby(group_col)["demand_qty"].shift(lag)

    for window in [7, 14, 28]:
        df[f"roll_mean_{window}"] = (
            df.groupby(group_col)["demand_qty"]
            .shift(1)
            .rolling(window=window, min_periods=1)
            .mean()
        )

    for window in [7, 28]:
        df[f"roll_std_{window}"] = (
            df.groupby(group_col)["demand_qty"]
            .shift(1)
            .rolling(window=window, min_periods=1)
            .std()
        )

    df["diff_1"] = df.groupby(group_col)["demand_qty"].diff(1)
    df["diff_7"] = df.groupby(group_col)["demand_qty"].diff(7)

    return df

def add_product_features(df: pd.DataFrame, group_col: str) -> pd.DataFrame:
    df = df.copy()
    product_demand = df.groupby(group_col)["demand_qty"].sum().rename("product_popularity")
    product_revenue = df.groupby(group_col)["revenue"].sum().rename("product_revenue_total")
    total_revenue = df["revenue"].sum()

    df = df.merge(product_demand, on=group_col, how="left")
    df = df.merge(product_revenue, on=group_col, how="left")
    df["product_revenue_share"] = df["product_revenue_total"] / total_revenue

    first_sale = df.groupby(group_col)["date"].min().rename("first_sale_date")
    df = df.merge(first_sale, on=group_col, how="left")
    df["product_lifecycle_age"] = (df["date"] - df["first_sale_date"]).dt.days

    df = df.drop(columns=["product_revenue_total", "first_sale_date"])
    return df

def build_time_series_dataframe(df: pd.DataFrame) -> pd.DataFrame:
    df_ts = df[["stock_code", "date", "demand_qty", "revenue", "num_invoices"]].copy()
    df_ts = add_time_features(df_ts)
    df_ts = df_ts[[
        "stock_code",
        "date",
        "demand_qty",
        "revenue",
        "num_invoices",
        "day_of_week",
        "week_of_year",
        "month",
        "is_weekend",
    ]]
    return df_ts

def build_tabular_dataframe(df: pd.DataFrame) -> pd.DataFrame:
    df_tab = add_time_features(df)
    df_tab = add_lag_rolling_features(df_tab, group_col="stock_code")
    df_tab["avg_price"] = df_tab["revenue"] / df_tab["demand_qty"].replace(0, np.nan)
    df_tab = add_product_features(df_tab, group_col="stock_code")
    return df_tab

daily_product_ts = build_time_series_dataframe(daily_product)
daily_product_tabular = build_tabular_dataframe(daily_product)

daily_product_ts.head()

In [ ]:
daily_product_ts.shape

In [ ]:
daily_product_tabular.tail()

In [ ]:
daily_product_tabular['stock_code'].value_counts()

In [ ]:
daily_product_tabular.shape

## Fitur untuk Modeling (Pasti Tersedia dari Dataset)\n
**Time Series DataFrame (`daily_product_ts`):**\n
- `stock_code`, `date` sebagai identitas seri dan index waktu\n
- `demand_qty` sebagai target\n
- `revenue`, `num_invoices` sebagai fitur tambahan\n
- `day_of_week`, `week_of_year`, `month`, `is_weekend` sebagai fitur waktu\n
\n
**Tabular DataFrame (`daily_product_tabular`):**\n
- Fitur dasar: `stock_code`, `date`, `demand_qty`, `revenue`, `num_invoices`\n
- Fitur waktu: `day_of_week`, `week_of_year`, `month`, `quarter`, `day_of_month`, `is_weekend`, `is_month_start`, `is_month_end`\n
- Lag & rolling: `lag_1`, `lag_7`, `lag_14`, `lag_28`, `roll_mean_7`, `roll_mean_14`, `roll_mean_28`, `roll_std_7`, `roll_std_28`, `diff_1`, `diff_7`\n
- Harga: `avg_price` (revenue / demand_qty)\n
- Fitur produk: `product_popularity`, `product_revenue_share`, `product_lifecycle_age`